## Importing Libraries

In [1]:
from datasets import load_dataset, get_dataset_split_names, get_dataset_config_names
from datasets import load_dataset_builder   # to inspect dataset (without actually downloading it)
#pip install transformers -U
#from transformers import AutoTokenizer

## Inspecting Dataset

**Reference**:
_MTS, Department of War UAP Release 1 — structured corpus, 2026. CC-BY-4.0._
This dataset is a structured, machine-readable companion to the source material at [war.gov/UFO/](https://www.war.gov/UFO/).

In [49]:
ufo_dataset = "MTSLIVE/war-gov-uap-release-1"
get_dataset_config_names(ufo_dataset)   # dataset files

['documents', 'pages', 'figures', 'videos']

We will use the `pages` file.

In [4]:
ds_ufo_builder = load_dataset_builder(ufo_dataset, "pages")
ds_ufo_builder.info.features

{'document_id': Value('string'),
 'page_no': Value('int64'),
 'text': Value('string'),
 'has_figures': Value('bool')}

In [61]:
pages = load_dataset(ufo_dataset, "pages", split="train")
pages

Dataset({
    features: ['document_id', 'page_no', 'text', 'has_figures'],
    num_rows: 4239
})

In [62]:
# example of what we are going to use
print(pages[0]['text'])

HEADQUARTERS
AIR MATERIEL COMMAND
WRIGHT FIELD, DAYTON, OHIO

DEC 1 9 1947

SUBJECT: Flying Discs

TO: Chief of Staff
United States Air Force
Washington 25, D. C.
ATTENTION: Director, Research & Development
Major General L. C. Craigie

1. Confirming the recent conversation of the undersigned with Major General L. C. Craigie, 9 December 1947, attached as listed below are copies of the reports from this Headquarters concerning Flying Discs.

2. Comments of Headquarters, Air Force on these letters have never been received by this Command. Continued and recent reports from qualified observers concerning this phenomenon still makes this matter one of concern to Headquarters, Air Materiel Command. Intelligence Department of this Command is continuing the collection and analysis of all available reports.

FOR THE COMMANDING GENERAL:

H. M. McCOY
Colonel, USAF
Chief of Intelligence

2 Attach:
cc ltr to CG, AAF, dtd 23 Sept 47 subj "AMC Opinion Concerning "Flying Discs""
cc ltr to CG, AAF, dtd 

## Check some documents by id

In [58]:
# all documents, each of these is composed of 1 or more pages
set(pages["document_id"])

{'18-100754-general-1946-7-vol-2',
 '18-6369445-general-1948-vol-1',
 '255-413270-ufo-s-and-defense-what-should-we-prepare-for',
 '331-120752-numeric-files-1944-1945-37153-german-armament-equipment-documents',
 '341-110448-records-relating-to-the-collection-and-dissemination-of-intelligence-1948-1955-ts-cont-no-2-2-5300-2-5399',
 '341-110677-numerical-file-5-2500',
 '342-hs1-416511228-319-1-flying-discs-1949',
 '38-143685-box-incident-summaries-101-172',
 '38-143685-box-incident-summaries-173-233',
 '38-143685-box7-incident-summaries-1-100',
 '59-214434-sp-16-7-18-1963',
 '59-64634-711-5612-7-2852',
 '65-hs1-101634279-100-de-18221-serial-844',
 '65-hs1-101634279-100-de-26505',
 '65-hs1-834228961-62-hq-83894-section-1',
 '65-hs1-834228961-62-hq-83894-section-10',
 '65-hs1-834228961-62-hq-83894-section-2',
 '65-hs1-834228961-62-hq-83894-section-3',
 '65-hs1-834228961-62-hq-83894-section-4',
 '65-hs1-834228961-62-hq-83894-section-5',
 '65-hs1-834228961-62-hq-83894-section-6',
 '65-hs1-834

Let's see if some pages have less than 20 characters.

In [64]:
null_id = 0     # count for document_id (even if it's just one page)
null_pgs = 0    # count for pages (can share the same id)
doc_ids = []    # to filter later(?)

for page in pages:
    if len(page["text"]) <= 20:
        null_pgs+=1
        if page["document_id"] not in doc_ids:
            null_id+=1
            doc_ids.append(page["document_id"])
        #print(f"doc_id: {page["document_id"]}\n text: {page["text"]}", "\n")

print(f"total null ids (at least one page): {null_id}")
print(f"total null pages: {null_pgs}")
print(f"problematic ids: {doc_ids}")

total null ids (at least one page): 61
total null pages: 225
problematic ids: ['342-hs1-416511228-319-1-flying-discs-1949', '65-hs1-101634279-100-de-26505', '65-hs1-834228961-62-hq-83894-section-1', '65-hs1-834228961-62-hq-83894-section-10', '65-hs1-834228961-62-hq-83894-section-2', '65-hs1-834228961-62-hq-83894-section-3', '65-hs1-834228961-62-hq-83894-section-4', '65-hs1-834228961-62-hq-83894-section-5', '65-hs1-834228961-62-hq-83894-section-6', '65-hs1-834228961-62-hq-83894-section-7', '65-hs1-834228961-62-hq-83894-section-8', '65-hs1-834228961-62-hq-83894-section-9', '65-hs1-834228961-62-hq-83894-serial-130', '65-hs1-834228961-62-hq-83894-serial-164', '65-hs1-834228961-62-hq-83894-serial-438', 'dow-uap-d10-mission-report-middle-east-may-2022', 'dow-uap-d12-mission-report-iraq-may-2022', 'dow-uap-d25-mission-report-greece-january-2024', 'dow-uap-d27-mission-report-united-arab-emirates-october-2023', 'dow-uap-d28-mission-report-iraq-september-2024', 'dow-uap-d3-mission-report-arabian

So there are 61 `document_id` with _at least_ one page with less than 20 characters. If we talk in terms of pages, there are 225 pages almost empty.

In [ ]:
# Here we inspect the "fbi-photo-ax" that for sure are photos

# select IDS
ID = [f"fbi-photo-a{i}" for i in range(1,9)] + [f"fbi-photo-b{i}" for i in range(1, 25)]

# filter
docs = pages.filter(lambda x: x["document_id"] in ID)
# docs = pages.filter(lambda x: x["has_figures"])

In [48]:
# display content
for doc in docs:
    # some doc may comprise more than 1 page
    if type(doc) == list:
        for page in doc:
            for key, item in page.items(): print(f"{key}: {item}\n")
    # if the doc has only one page
    elif type(doc) == dict:
        for key, item in doc.items(): print(f"{key}: {item}\n")
    else: raise RuntimeError("ziopera")

document_id: fbi-photo-a1

page_no: 1

text: 5

has_figures: True

document_id: fbi-photo-a2

page_no: 1

text: 5

has_figures: True

document_id: fbi-photo-a3

page_no: 1

text: 5

has_figures: True

document_id: fbi-photo-a4

page_no: 1

text: 

has_figures: True

document_id: fbi-photo-a5

page_no: 1

text: 5

has_figures: True

document_id: fbi-photo-a6

page_no: 1

text: 5


has_figures: True

document_id: fbi-photo-a7

page_no: 1

text: 5

has_figures: True

document_id: fbi-photo-a8

page_no: 1

text: 5

has_figures: True

document_id: fbi-photo-b1

page_no: 1

text: 12/31/99 18:11:19

has_figures: True

document_id: fbi-photo-b10

page_no: 1

text: 15 10 5 5 10 15
12/31/99 18:10:50

has_figures: True

document_id: fbi-photo-b11

page_no: 1

text: 12/31/99 18:11:06

has_figures: True

document_id: fbi-photo-b12

page_no: 1

text: 12/31/99 18:11:12

has_figures: True

document_id: fbi-photo-b13

page_no: 1

text: 12/31/99 18:19:54

has_figures: True

document_id: fbi-photo-b14

p